# C1 — Dự báo Doanh số Q2/2026
**DATA EXPLORERS 2026 | Xe đạp Thống Nhất**

Notebook này xây dựng mô hình dự báo doanh số bán xe đạp Thống Nhất cho Quý 2/2026 (tháng 4, 5, 6/2026) dựa trên dữ liệu lịch sử 15 tháng từ T1/2025 đến T3/2026.

**Mô hình sử dụng:** Prophet · SARIMAX · Ensemble (60/40)

**Thứ tự chạy:** Run All (tuần tự từ trên xuống)

## 1. Import Thư viện
> Nhập toàn bộ thư viện cần thiết. Chạy cell này đầu tiên.

In [ ]:
# ── Thư viện chuẩn ──────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ── Thống kê & kiểm định ────────────────────────────────────────────────────
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

# ── Mô hình dự báo ──────────────────────────────────────────────────────────
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
import pmdarima as pm

print("✅ Import thư viện thành công")

## 2. Wrangle Data (Tải & Chuẩn hóa Dữ liệu)
> Tải dữ liệu từ CSV, đổi tên cột cho đúng định dạng mô hình, chuẩn hóa kiểu dữ liệu.
>
> **Lưu ý về dữ liệu:**
> - T1–T3/2025 và T1–T3/2026: dữ liệu thực từ hệ thống bán hàng
> - T4–T12/2025: nội suy tuyến tính (ước tính) do dữ liệu gốc chưa đầy đủ

In [ ]:
# ── Tải dữ liệu lịch sử ─────────────────────────────────────────────────────
df = pd.read_csv("data/tnbike_data.csv", parse_dates=["ds"])

# ── Đổi tên cột cho Prophet (yêu cầu cột 'ds' và 'y') ────────────────────────
df = df.rename(columns={"revenue": "y"})
df["ds"] = pd.to_datetime(df["ds"])

# ── Tải dữ liệu kỳ dự báo Q2/2026 ───────────────────────────────────────────
future_df = pd.read_csv("data/future.csv", parse_dates=["ds"])
future_df["ds"] = pd.to_datetime(future_df["ds"])

print(f"Dữ liệu lịch sử  : {df.shape[0]} tháng")
print(f"Khoảng thời gian  : {df.ds.min().strftime('%m/%Y')} → {df.ds.max().strftime('%m/%Y')}")
print(f"Kỳ dự báo         : {future_df.ds.min().strftime('%m/%Y')} → {future_df.ds.max().strftime('%m/%Y')}")
df.head()

## 3. Kiểm tra Dữ liệu (Data Checks)
> Kiểm tra giá trị thiếu, kiểu dữ liệu, thống kê mô tả và tính hợp lệ trước khi mô hình hóa.

In [ ]:
# ── Hình dạng và kiểu dữ liệu ────────────────────────────────────────────────
print(f"Shape: {df.shape}")
print(f"\nKiểu dữ liệu:")
print(df.dtypes)

In [ ]:
# ── Kiểm tra giá trị thiếu ───────────────────────────────────────────────────
missing = df.isnull().sum()
print("Giá trị thiếu:")
print(missing)
if missing.sum() == 0:
    print("\n✅ Không có giá trị thiếu")
else:
    print(f"\n⚠️  Cần xử lý {missing.sum()} giá trị thiếu")

In [ ]:
# ── Thống kê mô tả ───────────────────────────────────────────────────────────
print("Thống kê doanh số (tỷ VND):")
print((df["y"] / 1e9).describe().round(2))
print(f"\nThống kê số lượng (chiếc):")
print(df["qty"].describe().round(0))

## 4. Phân tích Khám phá Dữ liệu (EDA)
> Trực quan hóa xu hướng doanh số, phân tích mùa vụ và kiểm định tính dừng của chuỗi thời gian.

In [ ]:
# ── Biểu đồ doanh số & số lượng theo tháng ──────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# doanh số
ax1.plot(df.ds, df.y / 1e9, marker="o", linewidth=2.5)
ax1.fill_between(df.ds, df.y / 1e9, alpha=0.15)
ax1.set_ylabel("Doanh số (tỷ VND)")
ax1.set_title("Doanh số & Số lượng bán — Xe đạp Thống Nhất (T1/2025–T3/2026)", fontsize=13, fontweight="bold")
ax1.grid(axis="y", alpha=0.4)

# số lượng
ax2.bar(df.ds, df.qty, width=20, alpha=0.75)
ax2.set_ylabel("Số lượng (chiếc)")
ax2.set_xlabel("Tháng")
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%m/%Y"))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha="right")
ax2.grid(axis="y", alpha=0.4)

fig.tight_layout()
plt.show()

In [ ]:
# ── Phân tích xu hướng & mùa vụ (Seasonal Decomposition) ────────────────────
decomp = seasonal_decompose(df.set_index("ds")["y"], model="additive", period=3)
fig = decomp.plot()
fig.set_size_inches(14, 10)
fig.suptitle("Phân tích Xu hướng & Mùa vụ — Doanh số Thống Nhất Bike", fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# ── Kiểm định ADF (Augmented Dickey-Fuller) — Tính dừng ─────────────────────
# Chuỗi gốc
pval_orig = adfuller(df["y"].dropna())[1]
# Sau sai phân bậc 1
pval_diff = adfuller(df["y"].diff().dropna())[1]

print("Kiểm định ADF (H0: chuỗi không dừng)")
print(f"  Chuỗi gốc  : p = {pval_orig:.4f}  → {'✅ DỪNG' if pval_orig < 0.05 else '⚠️  KHÔNG DỪNG — cần sai phân'}")
print(f"  Bậc sai phân 1: p = {pval_diff:.4f}  → {'✅ DỪNG' if pval_diff < 0.05 else '❌ VẪN KHÔNG DỪNG'}")

## 5. Xây dựng Pipeline Dữ liệu
> Chuẩn bị dữ liệu đầu vào cho mô hình: tách tập huấn luyện, xây dựng dataframe ngày lễ, gộp lịch sử + tương lai.

In [ ]:
# ── Tách tập huấn luyện ───────────────────────────────────────────────────────
training = df.copy()

# ── Gộp lịch sử + kỳ dự báo ─────────────────────────────────────────────────
future_renamed = future_df.rename(columns={"revenue": "y"}) if "revenue" in future_df.columns else future_df.copy()
df_full = pd.concat([df, future_renamed], ignore_index=True)
df_full["ds"] = pd.to_datetime(df_full["ds"])

# ── Kỳ dự báo Q2/2026 ────────────────────────────────────────────────────────
months_q2 = pd.date_range("2026-04-01", periods=3, freq="MS")
months_label = ["T4/2026", "T5/2026", "T6/2026"]

print(f"Tập huấn luyện : {len(training)} tháng")
print(f"Kỳ dự báo      : {[m.strftime('%m/%Y') for m in months_q2]}")

In [ ]:
# ── Xây dựng bảng ngày lễ (Holidays) ────────────────────────────────────────
# Tết Nguyên Đán
tet_dates = pd.to_datetime(df_full[df_full.is_tet == 1]["ds"])
tet = pd.DataFrame({
    "holiday": "tet_nguyen_dan",
    "ds": tet_dates,
    "lower_window": -7,
    "upper_window": 7,
})

# Ngày Thiếu nhi 1/6
thieu_nhi_dates = pd.to_datetime(df_full[df_full.is_thieu_nhi == 1]["ds"])
thieu_nhi = pd.DataFrame({
    "holiday": "thieu_nhi",
    "ds": thieu_nhi_dates,
    "lower_window": -3,
    "upper_window": 3,
})

holidays = pd.concat([tet, thieu_nhi], ignore_index=True)

# Bỏ cột flags khỏi tập huấn luyện (Prophet dùng holidays riêng)
drop_cols = [c for c in ["is_tet", "is_thieu_nhi"] if c in training.columns]
training = training.drop(columns=drop_cols)

print(f"✅ Đã xây dựng bảng ngày lễ: {len(holidays)} sự kiện")
print(holidays)

In [ ]:
# ── Tải tham số tốt nhất (từ Parameter Tuning) ───────────────────────────────
params_prophet = pd.read_csv("best_params/best_params_prophet.csv", index_col=0)
seasonality_mode        = str(params_prophet.loc["seasonality_mode"][0])
seasonality_prior_scale = float(params_prophet.loc["seasonality_prior_scale"][0])
holidays_prior_scale    = float(params_prophet.loc["holidays_prior_scale"][0])
changepoint_prior_scale = float(params_prophet.loc["changepoint_prior_scale"][0])

print("Tham số Prophet:")
print(f"  seasonality_mode        = {seasonality_mode}")
print(f"  seasonality_prior_scale = {seasonality_prior_scale}")
print(f"  holidays_prior_scale    = {holidays_prior_scale}")
print(f"  changepoint_prior_scale = {changepoint_prior_scale}")

## 6. Mô hình Prophet
> Huấn luyện mô hình Prophet với tham số đã chọn, dự báo Q2/2026 và đánh giá qua Cross-Validation.

In [ ]:
# ── Huấn luyện Prophet ───────────────────────────────────────────────────────
m = Prophet(
    holidays                = holidays,
    seasonality_mode        = seasonality_mode,
    seasonality_prior_scale = seasonality_prior_scale,
    holidays_prior_scale    = holidays_prior_scale,
    changepoint_prior_scale = changepoint_prior_scale,
    yearly_seasonality      = True,
    weekly_seasonality      = False,
    daily_seasonality       = False,
)
m.fit(training)
print("✅ Prophet đã huấn luyện xong")

In [ ]:
# ── Dự báo Q2/2026 ───────────────────────────────────────────────────────────
future_prophet = m.make_future_dataframe(periods=3, freq="MS")
forecast_prophet = m.predict(future_prophet)

# Lấy kết quả Q2/2026
predictions_prophet = (
    forecast_prophet[["ds", "yhat", "yhat_lower", "yhat_upper"]]
    .tail(3)
    .reset_index(drop=True)
)
predictions_prophet["tháng"] = predictions_prophet["ds"].dt.strftime("T%m/%Y")
predictions_prophet["yhat_ty"] = (predictions_prophet["yhat"] / 1e9).round(2)

print("Dự báo Prophet Q2/2026:")
for _, r in predictions_prophet.iterrows():
    print(f"  {r['tháng']}: {r['yhat_ty']} tỷ VND  [{r['yhat_lower']/1e9:.2f} – {r['yhat_upper']/1e9:.2f}]")

In [ ]:
# ── Biểu đồ thành phần mô hình Prophet ──────────────────────────────────────
fig = m.plot_components(forecast_prophet)
fig.suptitle("Thành phần Mô hình Prophet — Xu hướng, Mùa vụ, Ngày lễ",
             fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# ── Đánh giá Prophet — Cross-Validation ──────────────────────────────────────
df_cv = cross_validation(m, horizon="90 days", period="30 days", initial="90 days")
metrics = performance_metrics(df_cv)
rmse = metrics["rmse"].mean()
mape = metrics["mape"].mean()

print("Đánh giá Prophet (Cross-Validation):")
print(f"  RMSE  : {rmse:,.0f} VND  ({rmse/1e9:.2f} tỷ)")
print(f"  MAPE  : {mape*100:.1f}%")

In [ ]:
# ── Biểu đồ RMSE theo horizon ────────────────────────────────────────────────
fig = plot_cross_validation_metric(df_cv, metric="rmse")
fig.suptitle("RMSE Prophet theo Horizon Dự báo", fontsize=12, fontweight="bold")
fig.tight_layout()
plt.show()

## 7. Mô hình SARIMAX
> Mô hình ARIMA theo mùa (Seasonal ARIMA with Exogenous variables). Dùng tham số đã tối ưu từ bước Parameter Tuning.
> **Tham số tốt nhất:** ARIMA(1,1,0)(0,1,1)[3]

In [ ]:
# ── Tải tham số SARIMAX ───────────────────────────────────────────────────────
params_s = pd.read_csv("best_params/best_params_sarimax.csv", index_col=0)
p, d, q = int(params_s.loc["p"][0]), int(params_s.loc["d"][0]), int(params_s.loc["q"][0])
P, D, Q = int(params_s.loc["P"][0]), int(params_s.loc["D"][0]), int(params_s.loc["Q"][0])
m_s = 3  # chu kỳ quý

print(f"ARIMA order         : ({p},{d},{q})")
print(f"Seasonal order      : ({P},{D},{Q})[{m_s}]")

# ── Huấn luyện SARIMAX ───────────────────────────────────────────────────────
model_s = pm.ARIMA(
    order           = (p, d, q),
    seasonal_order  = (P, D, Q, m_s),
    suppress_warnings = True,
)
model_s.fit(training["y"].values)
print("\n✅ SARIMAX đã huấn luyện xong")

In [ ]:
# ── Dự báo Q2/2026 ───────────────────────────────────────────────────────────
sarimax_vals = model_s.predict(n_periods=3)
predictions_sarimax = pd.DataFrame({
    "ds":   months_q2,
    "yhat": sarimax_vals,
})
predictions_sarimax["tháng"] = predictions_sarimax["ds"].dt.strftime("T%m/%Y")

print("Dự báo SARIMAX Q2/2026:")
for _, r in predictions_sarimax.iterrows():
    print(f"  {r['tháng']}: {r['yhat']/1e9:.2f} tỷ VND")

In [ ]:
# ── Biểu đồ Fitted vs Actual (SARIMAX) ──────────────────────────────────────
fitted_vals = model_s.predict_in_sample()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(training.ds, training.y / 1e9, marker="o", linewidth=2, label="Thực tế")
ax.plot(training.ds, fitted_vals / 1e9, linestyle="--", linewidth=2, label="SARIMAX Fitted")
ax.set_title("SARIMAX — Thực tế vs Fitted", fontsize=12, fontweight="bold")
ax.set_xlabel("Tháng")
ax.set_ylabel("Doanh số (tỷ VND)")
ax.legend()
ax.grid(axis="y", alpha=0.4)
fig.tight_layout()
plt.show()

## 8. Ensemble: Prophet 60% + SARIMAX 40%
> Kết hợp hai mô hình theo trọng số để tận dụng ưu điểm của cả hai:
> - **Prophet** mạnh về nắm bắt xu hướng dài hạn và ngày lễ
> - **SARIMAX** mạnh về mùa vụ ngắn hạn và tính thống kê

In [ ]:
# ── Kết hợp Ensemble ────────────────────────────────────────────────────────
prophet_q2 = predictions_prophet["yhat"].values
sarimax_q2 = predictions_sarimax["yhat"].values

# Trọng số: 60% Prophet + 40% SARIMAX
w_prophet, w_sarimax = 0.6, 0.4
ensemble_yhat = w_prophet * prophet_q2 + w_sarimax * sarimax_q2

ensemble = pd.DataFrame({
    "ds":      months_q2,
    "tháng":   months_label,
    "prophet": prophet_q2,
    "sarimax": sarimax_q2,
    "yhat":    ensemble_yhat,
    "yhat_lower": predictions_prophet["yhat_lower"].values,
    "yhat_upper": predictions_prophet["yhat_upper"].values,
})

print(f"Ensemble ({int(w_prophet*100)}% Prophet + {int(w_sarimax*100)}% SARIMAX):")
for _, r in ensemble.iterrows():
    print(f"  {r['tháng']}: {r['yhat']/1e9:.2f} tỷ  "
          f"(Prophet {r['prophet']/1e9:.2f}T | SARIMAX {r['sarimax']/1e9:.2f}T)")

## 9. Trực quan hóa Kết quả
> So sánh 3 mô hình và biểu đồ chuỗi thời gian đầy đủ (lịch sử + dự báo).

In [ ]:
# ── Biểu đồ cột so sánh 3 mô hình ────────────────────────────────────────────
x = np.arange(3)
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
b1 = ax.bar(x - width,   prophet_q2 / 1e9, width, label="Prophet",          alpha=0.85)
b2 = ax.bar(x,           sarimax_q2 / 1e9, width, label="SARIMAX",           alpha=0.85)
b3 = ax.bar(x + width, ensemble_yhat / 1e9, width, label="Ensemble (60/40)", alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(months_label)
ax.set_ylabel("Doanh số (tỷ VND)")
ax.set_title("So sánh Dự báo Q2/2026 — Prophet vs SARIMAX vs Ensemble",
             fontsize=13, fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.4)

# Nhãn trên cột
for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                f"{bar.get_height():.1f}T", ha="center", va="bottom", fontsize=9, fontweight="bold")

fig.tight_layout()
plt.show()

In [ ]:
# ── Biểu đồ đường — Lịch sử + Dự báo Ensemble ───────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

# Lịch sử
ax.plot(training.ds, training.y / 1e9, "o-", linewidth=2.5, label="Lịch sử thực tế", zorder=5)

# Dự báo từng mô hình
ax.plot(months_q2, prophet_q2 / 1e9,    "s--", linewidth=2, markersize=9, label="Prophet")
ax.plot(months_q2, sarimax_q2 / 1e9,    "^--", linewidth=2, markersize=9, label="SARIMAX")
ax.plot(months_q2, ensemble_yhat / 1e9, "D-",  linewidth=2.5, markersize=10, label="Ensemble (Chính thức)")

# Vùng tin cậy
ax.fill_between(months_q2,
                ensemble["yhat_lower"] / 1e9,
                ensemble["yhat_upper"] / 1e9,
                alpha=0.15, label="Khoảng tin cậy Ensemble")

ax.axvline(pd.Timestamp("2026-04-01"), linestyle=":", alpha=0.6, linewidth=1.5)
ax.text(pd.Timestamp("2026-04-01"), ax.get_ylim()[1] * 0.97, "  Bắt đầu dự báo", fontsize=9, color="gray")

ax.set_title("Dự báo Doanh số Q2/2026 — Xe đạp Thống Nhất", fontsize=13, fontweight="bold")
ax.set_xlabel("Tháng")
ax.set_ylabel("Doanh số (tỷ VND)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")
ax.legend(loc="upper left")
ax.grid(axis="y", alpha=0.4)
fig.tight_layout()
plt.show()

## 10. Xuất Kết quả
> Lưu file dự báo vào thư mục `output/` để sử dụng trong báo cáo.

In [ ]:
# ── Tạo thư mục output nếu chưa có ──────────────────────────────────────────
os.makedirs("output", exist_ok=True)

# ── Lưu kết quả ──────────────────────────────────────────────────────────────
predictions_prophet.to_csv("output/predictions_prophet.csv",  index=False)
predictions_sarimax.to_csv("output/predictions_sarimax.csv",  index=False)
ensemble.to_csv("output/predictions_ensemble_c1.csv",         index=False)

print("✅ Đã lưu kết quả vào output/")
print("   → predictions_prophet.csv")
print("   → predictions_sarimax.csv")
print("   → predictions_ensemble_c1.csv")

In [ ]:
# ── Tóm tắt kết quả cuối cùng ────────────────────────────────────────────────
print("=" * 60)
print("    DỰ BÁO DOANH SỐ Q2/2026 — XE ĐẠP THỐNG NHẤT")
print("=" * 60)
for _, r in ensemble.iterrows():
    print(f"\n  {r['tháng']}")
    print(f"    Ensemble (chính thức) : {r['yhat']/1e9:.2f} tỷ VND")
    print(f"    ├─ Prophet (60%)      : {r['prophet']/1e9:.2f} tỷ VND")
    print(f"    └─ SARIMAX  (40%)     : {r['sarimax']/1e9:.2f} tỷ VND")
print("\n" + "=" * 60)
print(f"  Tổng Q2/2026 (Ensemble): {ensemble['yhat'].sum()/1e9:.2f} tỷ VND")
print("=" * 60)